In [ ]:
# Momentum strategy on S&P 500 — notebook assembly
# Imports and parameters
import pandas as pd
import numpy as np
from datetime import date

# Data wrappers (findata)
from findata.sp500_composition import get_sp500_composition  # fileciteturn3all1
from findata.equity_prices import get_equity_prices            # fileciteturn3all2

# Global parameters
START_DATE = "2015-01-01"
END_DATE   = "2025-01-01"
LOOKBACK_DAYS = 30
N_LONG = 50
N_SHORT = 50
TRADING_DAYS_PER_YEAR = 252

# Utility: cross-sectional zscore
def cs_zscore(x: pd.Series):
    mu = x.mean(skipna=True)
    sd = x.std(ddof=0, skipna=True)
    return (x - mu) / sd if sd and np.isfinite(sd) and sd > 0 else x * np.nan

In [ ]:
# 1) Universe: point-in-time S&P 500 members at START_DATE
sp500_tickers = get_sp500_composition(START_DATE)
len(sp500_tickers), sp500_tickers[:10]

In [ ]:
# 2) Fetch daily adjusted close prices for the universe
prices_all = get_equity_prices(
    tickers=sp500_tickers,
    start_date=START_DATE,
    end_date=END_DATE,
    fields=["Close"],
    frequency="1d",
    auto_adjust=True,
)
# Extract close prices as a simple DataFrame (rows=dates, cols=tickers)
close = prices_all["Close"].sort_index()
close.head()

In [ ]:
# 3) Build momentum signal: z-score of average returns over last 30 days (cross-sectional each day)
# Daily simple returns
ret = close.pct_change()
# Rolling average of returns over lookback window
avg30 = ret.rolling(LOOKBACK_DAYS, min_periods=LOOKBACK_DAYS//2).mean()
# Cross-sectional z-score per day
signal = avg30.apply(cs_zscore, axis=1)
signal.tail()

In [ ]:
# 4) Construct long-short equal-weight portfolio: top 50 long, bottom 50 short by signal
weights_long_sum = 1.0
weights_short_sum = -1.0

# Prepare empty weights DataFrame
raw_weights = pd.DataFrame(0.0, index=signal.index, columns=signal.columns)

# Function to set daily weights given one day's signal vector
def _make_daily_weights(sig_row: pd.Series) -> pd.Series:
    s = sig_row.dropna()
    if s.empty:
        return pd.Series(0.0, index=sig_row.index)
    longs = s.nlargest(N_LONG).index if len(s) >= N_LONG else s.sort_values(ascending=False).index
    shorts = s.nsmallest(N_SHORT).index if len(s) >= N_SHORT else s.sort_values(ascending=True).index
    w_long = (weights_long_sum / max(len(longs), 1))
    w_short = (weights_short_sum / max(len(shorts), 1))
    out = pd.Series(0.0, index=sig_row.index, dtype=float)
    out.loc[longs] = w_long
    out.loc[shorts] = w_short
    return out

raw_weights = signal.apply(_make_daily_weights, axis=1)
raw_weights.head()

In [ ]:
# 5) Apply 1-day execution lag to avoid lookahead; fill initial NaNs with 0
asset_weights = raw_weights.shift(1).fillna(0.0)
# Optional: ensure weights sum to zero each day (dollar-neutral); small numerical drift fix
row_sums = asset_weights.sum(axis=1)
if not np.allclose(row_sums.values, 0.0):
    asset_weights = asset_weights.sub(row_sums.values.reshape(-1, 1) / asset_weights.shape[1], axis=0)

asset_weights.head()

In [ ]:
# 6) Compute asset daily returns aligned to weights index and columns
asset_returns = close.pct_change().reindex(asset_weights.index).loc[:, asset_weights.columns].fillna(0.0)
asset_returns.head()